# Import libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import re

# Path

In [ ]:
input_path = 'drive/MyDrive/ViJudge/data/data_for_agreement/'

In [ ]:
vijudge_df = pd.read_csv(input_path + 'human_judge/ViJudge.csv')
vijudge_df.head()

## Preprocessing for Human Judges' Evaluations:

In [ ]:
vijudge_df_filtered = vijudge_df[~vijudge_df['question_id'].astype(str).str.startswith('1')]
vijudge_df_filtered.head()

## Processing for Model Judges' Evaluations:

In [ ]:
list_judges = ['openai/gpt-oss-20b', 'llama-3.3-70b-versatile', 'moonshotai/kimi-k2-instruct', 'qwen/qwen3-32b', 'deepseek/deepseek-chat-v3.1']

In [ ]:
def extract_model_name(model_string):
  match = re.match(r'[^/-]+', model_string)
  if match:
    return match.group(0)
  else:
    return model_string

In [ ]:
def is_self_preferred(row, llmjudge_name):
    model_a_name = extract_model_name(row['model_a'])
    model_b_name = extract_model_name(row['model_b'])

    if (row['verdict'] == 'A' and model_a_name == llmjudge_name) or \
       (row['verdict'] == 'B' and model_b_name == llmjudge_name):
        return True
    return False

In [ ]:
dict_llms = {
    1: ['openai'],
    2: ['llama'],
    3: ['moonshotai'],
    4: ['qwen'],
    5: ['deepseek']
}

## No Tie:

In [ ]:
# Remove tie evaluations from human judges
vijudge_df_filtered_no_tie = vijudge_df_filtered[vijudge_df_filtered['verdict'] != 'C']
vijudge_pairs_no_tie = vijudge_df_filtered_no_tie.groupby('question_id')['verdict'].apply(list).reset_index()
vijudge_pairs_no_tie = vijudge_pairs_no_tie.rename(columns={'verdict': 'human_verdicts'})
vijudge_pairs_no_tie.head()

In [ ]:
judges = []
bias = []

for i in dict_llms:
  print(f'Measure self-preference bias for: {dict_llms[i]}')
  llmjudges_df = pd.read_csv(input_path + 'LLM_judge/' + dict_llms[i][0] + '_pair.csv')

  # Filter out evaluations where llmjudge_i itself is a candidate
  # The number of llmjudges is 5
  # Therefore, the number of samples here is 400 (4 pairs where llmjudge_i is a candidate)
  llmjudge_df = llmjudges_df[
      (llmjudges_df['model_a'].apply(extract_model_name) == dict_llms[i][0]) |
      (llmjudges_df['model_b'].apply(extract_model_name) == dict_llms[i][0])
  ]

  # Exclude evaluations by llmjudge_i with a verdict of "Unknown"
  llmjudge_df = llmjudge_df[llmjudge_df['verdict'] != 'Unknown']

  # Exclude evaluations by llmjudge_i with a verdict of "C"
  llmjudge_df_no_tie = llmjudge_df[llmjudge_df['verdict'] != 'C']

  # Merge the evaluation DataFrames from llmjudge_i and humanjudge, keeping all question_ids present in both
  merged_df = llmjudge_df_no_tie.merge(vijudge_pairs_no_tie, on='question_id', how='inner')

  # Exclude evaluations where llmjudge_i matches at least one human judge
  llmjudge_verdict_filtered = merged_df[
      ~merged_df.apply(lambda row: row['verdict'] in row['human_verdicts'], axis=1)
  ]
  columns_to_show = ['question_id', 'model_a', 'model_b', 'verdict', 'human_verdicts']
  display(llmjudge_verdict_filtered[columns_to_show].sample(5))

  # Count evaluations that differ from human judges and where llmjudge_i prefers its own response
  self_preference_count = llmjudge_verdict_filtered.apply(lambda row: is_self_preferred(row, dict_llms[i][0]), axis=1).sum()

  print(f"Number of rows where [{dict_llms[i][0]}] shows self-preference: {self_preference_count}")
  print(f"Total rows where [{dict_llms[i][0]}] differs from humans: {len(llmjudge_verdict_filtered)}")
  print(f"Percentage of self-preference: {self_preference_count / len(llmjudge_verdict_filtered) * 100:.2f}%")

  judges.append(dict_llms[i][0])
  bias.append(self_preference_count / len(llmjudge_verdict_filtered) * 100)
  print('#'*30)

bias_no_tie = pd.DataFrame({'judge': judges, 'bias (%)': bias})

# Save & show result

In [ ]:
print('bias without tie:')
bias_no_tie.to_csv('drive/MyDrive/ViJudge/data/self_preference_bias/statistics/bias_result.csv', index=False)
bias_no_tie